# v27 — Bedah Menyeluruh 212 LOSS v13 (Cari Benang Merah, Bukan Cari 1 Filter)

**Latar belakang & pergeseran pendekatan:** v25 (ADX ceiling/exhaustion) dan v26 (RSI
multi-timeframe exhaustion) sama-sama GAGAL menemukan filter yang benar2 memperbaiki performa
-- keduanya cuma kenaikan PF ~0.01-0.02 di TRAIN, nyaris tidak signifikan (kemungkinan noise
statistik). Pola "RSI overbought bersamaan multi-timeframe" yang kelihatan jelas di 3 trade
live spesifik TERNYATA TIDAK terbukti sbg pola sistematis di 2282 trade TRAIN (v26 Section 4:
selisih RSI WIN vs LOSS cuma 0.5-2.0 poin, nyaris tidak ada beda).

**User secara eksplisit reframe tujuan riset**: trading tidak harus selalu profit per-trade --
yang penting kerugian bisa "ketutup" oleh kemenangan (profit factor tetap sehat), BUKAN
menghilangkan loss sepenuhnya. Jadi drpd terus mencari 1 filter ajaib (pendekatan yang sudah
2x gagal), riset ini **PETAKAN dulu SEMUA kemungkinan pola di 212 LOSS v13** (dari trade log
backtest 721 trade, `trade_log_full_final.csv`) scr deskriptif menyeluruh -- baru putuskan mana
yang benar2 punya benang merah kuat vs mana yang cuma noise/acak.

**Konteks angka dasar**: 721 trade, 509 WIN ($7080.11 total) vs 212 LOSS (-$1934.84 total),
PF=3.66, rasio avg win:avg loss = 1.52:1. Sistem SEHAT secara agregat -- pertanyaannya BUKAN
"kenapa strategi ini rugi" (dia tidak, secara total untung besar), tapi "apakah 212 LOSS ini
py pola yg predictable, atau memang variasi acak yang wajar dlm strategi manapun".

**Cakupan bedah (menyeluruh, bukan 1 sudut pandang)**:
1. Breakdown LOSS by exit_reason (SL murni vs TIMEOUT) -- beda mekanisme kegagalan
2. Breakdown by jam/sesi trading
3. Breakdown by ADX & momentum chain saat entry
4. Breakdown by arah (BUY vs SELL)
5. Breakdown by ukuran ATR/volatilitas saat entry
6. Losing streak analysis -- seberapa sering & seberapa panjang rentetan loss terjadi, dan
   apakah rentetan itu selalu "ketutup" oleh WIN berikutnya (cek recovery time)
7. Precision/quality check: apakah trade yang exit_reason=TIMEOUT itu "hampir menang" (harga
   dekat TP tapi kehabisan waktu) atau benar2 flat/salah arah

**TIDAK ADA perubahan ke `usecase.py`** -- murni analisis deskriptif, dasar utk keputusan
lanjutan (baik itu "tidak perlu diubah, sistem sudah sehat" atau "ini spesifik layak diperbaiki").

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

STRATEGY_NAME = "m5_scalping"
VERSION = "v27"

PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed" / STRATEGY_NAME
EXPORT_DIR = PROJECT_ROOT / "dataset" / "exports" / STRATEGY_NAME / VERSION
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 180)
plt.rcParams["figure.figsize"] = (14, 5)

## 1. Load trade log v13 final (721 trade) & ringkasan dasar

In [2]:
df = pd.read_csv(PROCESSED_DIR / "v13" / "trade_log_full_final.csv")
df["entry_time"] = pd.to_datetime(df["entry_time"])
df["exit_time"] = pd.to_datetime(df["exit_time"])
df["hour_utc"] = df["entry_time"].dt.hour
df["dominant_chain"] = df[["ind_bull_chain", "ind_bear_chain"]].max(axis=1)

def session(h):
    if 0 <= h < 7: return "Asia"
    if 7 <= h < 13: return "London"
    if 13 <= h < 21: return "NewYork"
    return "Sepi/Overlap-akhir"
df["session"] = df["hour_utc"].apply(session)

print(f"Total: {len(df)} trade, {df['entry_time'].min()} -> {df['entry_time'].max()}")
wins = df[df.result=="WIN"]
losses = df[df.result=="LOSS"]
print(f"WIN: {len(wins)} (${wins.pnl.sum():.2f}, avg ${wins.pnl.mean():.2f})")
print(f"LOSS: {len(losses)} (${losses.pnl.sum():.2f}, avg ${losses.pnl.mean():.2f})")
print(f"Profit Factor: {wins.pnl.sum()/abs(losses.pnl.sum()):.2f}")
print(f"\nLOSS by exit_reason:")
print(losses.exit_reason.value_counts())

Total: 721 trade, 2025-01-06 00:55:00+00:00 -> 2026-08-05 23:40:00+00:00
WIN: 509 ($7080.11, avg $13.91)
LOSS: 212 ($-1934.84, avg $-9.13)
Profit Factor: 3.66

LOSS by exit_reason:
exit_reason
SL         139
TIMEOUT     73
Name: count, dtype: int64


## 2. LOSS by jam/sesi trading

In [3]:
def loss_rate_by(df, group_col):
    rows = []
    for val, g in df.groupby(group_col, observed=True):
        loss_ct = (g.result=="LOSS").sum()
        rows.append({
            group_col: val, "n": len(g), "loss_n": loss_ct,
            "loss_rate_pct": round(loss_ct/len(g)*100, 1),
            "pf": round(g.loc[g.pnl>0,"pnl"].sum() / abs(g.loc[g.pnl<=0,"pnl"].sum()), 2) if (g.pnl<=0).any() else float("inf"),
            "net_pnl": round(g.pnl.sum(), 2),
        })
    return pd.DataFrame(rows).sort_values("loss_rate_pct", ascending=False)

print("=== Loss rate by session ===")
print(loss_rate_by(df, "session").to_string(index=False))
print()
print("=== Loss rate by hour (top 10 jam dengan loss rate tertinggi, min n=10) ===")
by_hour = loss_rate_by(df, "hour_utc")
print(by_hour[by_hour["n"]>=10].head(10).to_string(index=False))

=== Loss rate by session ===
           session   n  loss_n  loss_rate_pct   pf  net_pnl
           NewYork 243      79           32.5 3.79  1705.73
            London 161      52           32.3 2.71   892.25
Sepi/Overlap-akhir  58      16           27.6 2.56   442.42
              Asia 259      65           25.1 5.06  2104.87

=== Loss rate by hour (top 10 jam dengan loss rate tertinggi, min n=10) ===
 hour_utc  n  loss_n  loss_rate_pct   pf  net_pnl
        9 17       9           52.9 0.27   -82.59
       19 30      14           46.7 2.26   123.08
       16 23       9           39.1 4.14   155.95
       17 19       7           36.8 2.34    87.15
       11 25       9           36.0 2.33   172.74
       22 20       7           35.0 1.67    58.86
       10 26       9           34.6 2.51   119.40
        5 35      12           34.3 3.76   305.32
       20 24       8           33.3 3.05   134.91
        8 28       9           32.1 6.00   157.58


## 3. LOSS by ADX & momentum chain saat entry

In [4]:
df["adx_bucket"] = pd.cut(df["ind_adx"], bins=[0,20,25,30,35,40,50,100], labels=["<20","20-25","25-30","30-35","35-40","40-50","50+"])
print("=== Loss rate by ADX bucket ===")
print(loss_rate_by(df, "adx_bucket").to_string(index=False))
print()
print("=== Loss rate by dominant momentum chain ===")
print(loss_rate_by(df, "dominant_chain").to_string(index=False))
print()
print("=== Loss rate by chain_maxed (exhaustion mode aktif atau tidak) ===")
print(loss_rate_by(df, "chain_maxed").to_string(index=False))

=== Loss rate by ADX bucket ===
adx_bucket   n  loss_n  loss_rate_pct    pf  net_pnl
     25-30 146      48           32.9  3.24  1043.55
     40-50  71      23           32.4  2.26   332.14
     20-25 257      77           30.0  2.94  1428.55
     35-40  65      19           29.2  5.85   597.52
     30-35  86      23           26.7  4.53   770.75
       <20  61      14           23.0 13.94   694.83
       50+  35       8           22.9  4.79   277.93

=== Loss rate by dominant momentum chain ===
 dominant_chain   n  loss_n  loss_rate_pct   pf  net_pnl
            4.0  19       8           42.1 1.16    18.30
            5.0 100      39           39.0 3.11   649.46
            7.0 195      59           30.3 4.72  1726.89
            6.0 216      64           29.6 3.53  1616.82
            8.0 191      42           22.0 3.80  1133.80

=== Loss rate by chain_maxed (exhaustion mode aktif atau tidak) ===
 chain_maxed   n  loss_n  loss_rate_pct   pf  net_pnl
       False 530     170         

## 4. LOSS by arah (BUY vs SELL) & volatilitas (ATR) saat entry

In [5]:
print("=== Loss rate by direction ===")
print(loss_rate_by(df, "direction").to_string(index=False))
print()
df["atr_bucket"] = pd.qcut(df["atr_at_entry"], q=4, labels=["Q1 (rendah)","Q2","Q3","Q4 (tinggi)"])
print("=== Loss rate by ATR quartile saat entry ===")
print(loss_rate_by(df, "atr_bucket").to_string(index=False))

=== Loss rate by direction ===
direction   n  loss_n  loss_rate_pct   pf  net_pnl
      BUY 477     148           31.0 3.22  2469.04
     SELL 244      64           26.2 4.26  2676.23

=== Loss rate by ATR quartile saat entry ===
 atr_bucket   n  loss_n  loss_rate_pct   pf  net_pnl
Q1 (rendah) 181      60           33.1 2.29   308.85
         Q2 180      52           28.9 3.99  1264.00
         Q3 180      52           28.9 3.57  1588.31
Q4 (tinggi) 180      48           26.7 4.03  1984.11


## 5. Statistical significance check -- yang keliatan beda, beneran signifikan atau kebetulan sample kecil?

Breakdown di atas ada yang variasinya cukup besar -- tapi perlu dicek pakai uji statistik
(bukan cuma "kelihatan beda"), supaya tidak salah simpulkan pola dari fluktuasi sample kecil.

In [6]:
from scipy.stats import chi2_contingency

def chi2_test(df, group_col):
    ct = pd.crosstab(df[group_col], df["result"])
    chi2, p, dof, expected = chi2_contingency(ct)
    return p

for col in ["session", "adx_bucket", "dominant_chain", "chain_maxed", "direction", "atr_bucket"]:
    p = chi2_test(df, col)
    flag = " <-- SIGNIFIKAN (p<0.05)" if p < 0.05 else ""
    print(f"{col:<18} chi2 p-value = {p:.4f}{flag}")

session            chi2 p-value = 0.2420
adx_bucket         chi2 p-value = 0.7529
dominant_chain     chi2 p-value = 0.0261 <-- SIGNIFIKAN (p<0.05)
chain_maxed        chi2 p-value = 0.0114 <-- SIGNIFIKAN (p<0.05)
direction          chi2 p-value = 0.2107
atr_bucket         chi2 p-value = 0.5894


## 6. Losing streak analysis -- seberapa panjang & apakah selalu "ketutup"

Fokus ke pertanyaan user: bukan soal menghindari loss, tapi apakah rentetan loss itu
terkelola (selalu ada cukup WIN sesudahnya utk menutup), atau ada periode dimana modal
beneran terancam.

In [7]:
df_sorted = df.sort_values("entry_time").reset_index(drop=True)

streaks = []
current_streak = 0
streak_start_idx = None
for idx, r in df_sorted.iterrows():
    if r["result"] == "LOSS":
        if current_streak == 0:
            streak_start_idx = idx
        current_streak += 1
    else:
        if current_streak > 0:
            streaks.append({
                "streak_length": current_streak,
                "start_time": df_sorted.loc[streak_start_idx, "entry_time"],
                "total_loss_in_streak": df_sorted.loc[streak_start_idx:idx-1, "pnl"].sum(),
            })
        current_streak = 0

streak_df = pd.DataFrame(streaks)
print("=== Distribusi panjang losing streak ===")
print(streak_df["streak_length"].value_counts().sort_index())
print()
print(f"Losing streak terpanjang: {streak_df['streak_length'].max()} trade berturut-turut")
print(f"Kerugian terbesar dalam 1 streak: \${streak_df['total_loss_in_streak'].min():.2f}")
print()

# Cek recovery: setelah tiap losing streak, berapa lama sampai equity balik ke level sblm streak dimulai
df_sorted["cum_pnl"] = df_sorted["pnl"].cumsum()
long_streaks = streak_df[streak_df["streak_length"] >= 3].copy()
print(f"=== Recovery time utk {len(long_streaks)} losing streak >= 3 trade berturut-turut ===")
for _, s in long_streaks.iterrows():
    streak_start_time = s["start_time"]
    idx_at_streak_start = df_sorted[df_sorted["entry_time"] == streak_start_time].index[0]
    equity_before_streak = df_sorted.loc[idx_at_streak_start - 1, "cum_pnl"] if idx_at_streak_start > 0 else 0
    after = df_sorted.loc[idx_at_streak_start:]
    recovered = after[after["cum_pnl"] >= equity_before_streak]
    if len(recovered) > 0:
        recovery_trades = recovered.index[0] - idx_at_streak_start + 1
        print(f"  Streak {int(s['streak_length'])}x @ {streak_start_time}: recovery dalam {recovery_trades} trade berikutnya")
    else:
        print(f"  Streak {int(s['streak_length'])}x @ {streak_start_time}: BELUM recovery sampai akhir data")

=== Distribusi panjang losing streak ===
streak_length
1    98
2    33
3     7
4     4
5     2
Name: count, dtype: int64

Losing streak terpanjang: 5 trade berturut-turut
Kerugian terbesar dalam 1 streak: \$-58.70

=== Recovery time utk 13 losing streak >= 3 trade berturut-turut ===
  Streak 3x @ 2025-03-19 22:30:00+00:00: recovery dalam 8 trade berikutnya
  Streak 3x @ 2025-04-22 03:20:00+00:00: recovery dalam 6 trade berikutnya
  Streak 5x @ 2025-05-06 18:30:00+00:00: recovery dalam 8 trade berikutnya
  Streak 4x @ 2025-07-10 07:45:00+00:00: recovery dalam 8 trade berikutnya
  Streak 3x @ 2025-08-14 14:40:00+00:00: recovery dalam 5 trade berikutnya
  Streak 4x @ 2025-10-02 03:25:00+00:00: recovery dalam 6 trade berikutnya
  Streak 3x @ 2025-10-06 07:45:00+00:00: recovery dalam 4 trade berikutnya
  Streak 5x @ 2025-10-09 08:35:00+00:00: recovery dalam 20 trade berikutnya
  Streak 4x @ 2026-02-17 15:40:00+00:00: recovery dalam 7 trade berikutnya
  Streak 3x @ 2026-02-25 02:20:00+00:00:

<>:25: SyntaxWarning: invalid escape sequence '\$'
<>:25: SyntaxWarning: invalid escape sequence '\$'
C:\Users\baus12345\AppData\Local\Temp\ipykernel_223632\1127739245.py:25: SyntaxWarning: invalid escape sequence '\$'
  print(f"Kerugian terbesar dalam 1 streak: \${streak_df['total_loss_in_streak'].min():.2f}")


## 7. Kualitas TIMEOUT loss -- apakah "hampir menang" atau beneran salah arah?

73 dari 212 LOSS itu exit_reason=TIMEOUT (bukan kena SL). Cek seberapa dekat harga ke TP saat
timeout terjadi -- kalau banyak yang "hampir sampai TP", itu indikasi max_hold_candles=12
mungkin terlalu pendek utk kondisi tsb; kalau jauh dari TP (net masih rugi/flat), itu beneran
salah arah/tidak berkembang.

In [8]:
timeout_losses = df[(df.result=="LOSS") & (df.exit_reason=="TIMEOUT")].copy()
print(f"Total TIMEOUT LOSS: {len(timeout_losses)}")

# Progress ke arah TP: (exit_price - entry_price) / (tp_price - entry_price), searah direction
def progress_to_tp(row):
    if row["direction"] == "BUY":
        total_dist = row["tp_price"] - row["entry_price"]
        actual_dist = row["exit_price"] - row["entry_price"]
    else:
        total_dist = row["entry_price"] - row["tp_price"]
        actual_dist = row["entry_price"] - row["exit_price"]
    return actual_dist / total_dist if total_dist != 0 else 0

timeout_losses["progress_to_tp_pct"] = timeout_losses.apply(progress_to_tp, axis=1) * 100
print()
print("=== Distribusi 'seberapa jauh ke arah TP' saat timeout (0%=di entry, 100%=nyaris TP, negatif=malah mundur) ===")
print(timeout_losses["progress_to_tp_pct"].describe())
print()
bins = [-200, -50, 0, 25, 50, 75, 100]
labels = ["<-50% (mundur jauh)", "-50-0% (mundur)", "0-25%", "25-50%", "50-75%", "75-100% (hampir TP)"]
timeout_losses["progress_bucket"] = pd.cut(timeout_losses["progress_to_tp_pct"], bins=bins, labels=labels)
print(timeout_losses["progress_bucket"].value_counts().sort_index())

Total TIMEOUT LOSS: 73

=== Distribusi 'seberapa jauh ke arah TP' saat timeout (0%=di entry, 100%=nyaris TP, negatif=malah mundur) ===
count     73.000000
mean     -18.803890
std       17.110062
min     -100.877877
25%      -29.095119
50%      -15.595684
75%       -5.143874
max        0.000000
Name: progress_to_tp_pct, dtype: float64

progress_bucket
<-50% (mundur jauh)     2
-50-0% (mundur)        71
0-25%                   0
25-50%                  0
50-75%                  0
75-100% (hampir TP)     0
Name: count, dtype: int64


## 8. Kesimpulan

*(diisi setelah lihat hasil eksekusi lengkap Section 1-7 -- placeholder)*